**Eye-Tracking & Neurocognitive Intelligence | Eye-Tracking Foundations & Signal Analysis | Presented by: Reza Saadatyar (2026)**

**07 | Integrated Evidence and Reproducibility Review**

![Source-identified evidence integration from Notebooks 01–06 and bounded scientific conclusions](../../images/07_integrated_evidence_and_reproducibility.svg)

**Objective:** Combine **small saved summary files** from the six preceding notebooks, verify their record and participant correspondence, and provide a transparent scientific overview without reprocessing original gaze, recomputing eye events or fitting a model.

**Observed upstream scope:** GazeBase v2.0, Round 1, Session 1; **10 participants, 40 source recordings and 2,778,008 original samples**. Notebook 03 reported **7,015 I-VT fixation**, **9,304 I-VT saccade** and **5,409 I-DT fixation** intervals. Notebook 04 produced 40 source-aligned feature rows. Notebook 05 reported 16 participant/task-feature statistics using ten independent people and an exploratory 500-draw bootstrap.

**Sensitivity evidence:** Notebook 06 evaluated **20, 30 and 40 dva/s** across 40 records. Its record-averaged conditional EyeLink agreements were approximately **90.97%, 98.13% and 98.78%**; **their eligible comparison denominators differ**. The new Notebook 06 support check must be executed before this integrated review.

**Scientific boundary:** Vendor annotations are not independent expert ground truth; event classes and derived features depend on provisional settings and extreme calculated gaze speeds. No clinical, cognitive, causal, real-time or population-generalisation conclusion is supported.

**Execution status:** Notebook 07 is prepared but has not yet executed on the source subset. Its introductory figure explicitly separates observed upstream results from the planned compact reconciliation.


**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("eye_tracking_foundations")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# ------------------------ 5. Study Research Utilities ------------------------
from core.stage_outputs import ensure_stage_inputs
from gaze_source import read_stage
from gaze_integration import (
    reconcile_saved_record_evidence, validate_participant_statistics,
    validate_threshold_summaries,
)

# --------------------- 6. Output and Presentation Setup ----------------------
output_dir = root / "eye_tracking_foundations" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
plot_style(
    font_family="DejaVu Sans", title_size=12.5,
    title_weight="normal", label_size=11.5,
    label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(10, 4),
)
print("Current outputs:", output_dir)


**02. Recover Named Current Evidence Tables**

Read only the current compact CSV outputs from Notebooks 01–06, restoring an explicitly named Notebook ZIP from `/content/` only when a required file is absent. No raw GazeBase CSV or large sample-level `01_`, `02_` or `03_` table is needed.

Inputs comprise source quality, Notebook 02's per-record preprocessing summary, Notebook 03's vendor comparison and event-class totals, Notebook 04's per-record features, Notebook 05's participant table and descriptive statistics, and Notebook 06's detailed threshold table, original record-mean summary and **new** denominator-support table.

The initial printed table lists which files were opened and their row counts. A matching filename alone does not establish that outputs belong to the same participant selection: the next section checks their actual identities and numerical relationships.


In [ ]:
# ================== 02. Recover Named Current Evidence ======================

# ------------------ 1. Restore Exact Current Notebook Outputs ----------------
REQUIRED = {
    1: ("01_quality_by_record.csv",),
    2: ("02_preprocessing_summary.csv",),
    3: ("03_vendor_agreement.csv", "03_event_class_summary.csv"),
    4: ("04_features_by_record.csv",),
    5: ("05_participant_features.csv", "05_descriptive_statistics.csv"),
    6: ("06_sensitivity.csv", "06_sensitivity_summary.csv",
        "06_reference_support.csv"),
}
for notebook_number, filenames in REQUIRED.items():
    ensure_stage_inputs(output_dir, notebook_number, filenames)

# -------------------------- 2. Load Compact Tables ---------------------------
quality = read_stage(
    output_dir / "01_quality_by_record.csv",
    ("record_id", "participant_id", "task", "n_samples", "valid_fraction"),
)
preprocessing = read_stage(
    output_dir / "02_preprocessing_summary.csv",
    ("record_id", "participant_id", "task", "n_samples"),
)
vendor = read_stage(
    output_dir / "03_vendor_agreement.csv",
    ("record_id", "eligible_samples", "sample_agreement"),
)
event_counts = read_stage(
    output_dir / "03_event_class_summary.csv",
    ("method", "event_type", "n_events"),
)
features = read_stage(
    output_dir / "04_features_by_record.csv",
    ("record_id", "participant_id", "task", "valid_fraction"),
)
participants = read_stage(
    output_dir / "05_participant_features.csv",
    ("task", "participant_id"),
)
stats = read_stage(
    output_dir / "05_descriptive_statistics.csv",
    ("task", "feature", "n_independent_participants", "participant_mean"),
)
sensitivity = read_stage(
    output_dir / "06_sensitivity.csv",
    ("record_id", "threshold_dva_s", "eligible_samples", "sample_agreement"),
)
threshold_summary = read_stage(
    output_dir / "06_sensitivity_summary.csv",
    ("threshold_dva_s", "n_records", "mean_predicted_fixation_fraction"),
)
reference_support = read_stage(
    output_dir / "06_reference_support.csv",
    ("threshold_dva_s", "n_records", "total_vendor_evaluable_samples"),
)

# -------------------- 3. Display Input Evidence Coverage ---------------------
input_coverage = pd.DataFrame([
    {"notebook": 1, "table": "Source quality", "rows": len(quality)},
    {"notebook": 2, "table": "Preprocessing", "rows": len(preprocessing)},
    {"notebook": 3, "table": "Vendor reference", "rows": len(vendor)},
    {"notebook": 3, "table": "Event classes", "rows": len(event_counts)},
    {"notebook": 4, "table": "Record features", "rows": len(features)},
    {"notebook": 5, "table": "Participant features", "rows": len(participants)},
    {"notebook": 5, "table": "Descriptive statistics", "rows": len(stats)},
    {"notebook": 6, "table": "Sensitivity (detailed)", "rows": len(sensitivity)},
    {"notebook": 6, "table": "Sensitivity (summary)", "rows": len(threshold_summary)},
    {"notebook": 6, "table": "Vendor support", "rows": len(reference_support)},
])
display(input_coverage)
print("Source participants in quality table:", quality["participant_id"].nunique())
print("Original gaze samples recorded:", f"{int(quality['n_samples'].sum()):,}")


**03. Reconcile Record, Participant and Statistical Evidence**

Enforce one-to-one correspondence among the source quality, preprocessing, vendor-comparison and feature tables using original `record_id` values. Check Notebook 01 versus Notebook 02 source sample counts, participant and task identities; compare valid fractions from Notebooks 01 and 04; verify the **30-dva/s I-VT reference** against Notebook 03.

Check that the Notebook 05 participant/task combinations correspond to the source features and that its `n_independent_participants` and participant means agree with the saved participant-level table. Rebuild Notebook 06's threshold denominator statistics from its 120-row detailed table and compare them with both saved compact summaries.

**No historical cohort-size assertion is used as a substitute for alignment.** If current producer files refer to different records, tasks or numerical results, stop with a specific error rather than reporting a combined result. Save `07_source_alignment.csv` to document successful record-level reconciliation.


In [ ]:
# ============== 03. Reconcile Recorded Evidence and Identities ===============

# ------------------- 1. Check Shared Record and Timebase Scope ---------------
alignment = reconcile_saved_record_evidence(
    quality, preprocessing, vendor, features, sensitivity,
)
alignment.to_csv(output_dir / "07_source_alignment.csv", index=False)
print("Aligned source recordings:", len(alignment))
print("Maximum Notebook 01/04 valid-fraction difference:",
      alignment["absolute_valid_fraction_difference"].max())

# ------------------- 2. Check Participant-Level Statistics -------------------
independent_people = validate_participant_statistics(
    features, participants, stats,
)
print("Independent participants represented:", independent_people)
print("Validated task-feature summary rows:", len(stats))

# ------------------- 3. Reconcile Threshold and Vendor Support ----------------
recomputed_support = validate_threshold_summaries(
    sensitivity, threshold_summary, reference_support,
)
print("Predefined I-VT settings:", recomputed_support.threshold_dva_s.tolist())
print("Eligible vendor samples by setting:")
display(recomputed_support[[
    "threshold_dva_s", "n_records", "n_vendor_evaluable_records",
    "total_vendor_evaluable_samples",
    "mean_conditional_vendor_agreement",
]])

# ------------------ 4. Preserve Original Detected Event Counts ---------------
if event_counts.duplicated(["method", "event_type"]).any():
    raise ValueError("Duplicate Notebook 03 event-class summaries")
if not (pd.to_numeric(event_counts["n_events"], errors="raise") >= 0).all():
    raise ValueError("Invalid Notebook 03 event counts")
display(event_counts)
print("Total recorded detector event intervals:",
      int(event_counts["n_events"].sum()))


**04. Construct an Evidence-Bounded Study Summary**

Create one machine-readable summary row describing **recorded source scope**, valid gaze fraction, independently represented people, number of algorithm-detected event intervals, feature-table coverage, exploratory participant statistics, vendor-reference availability and the number of fixed I-VT sensitivity settings.

Numbers are derived from saved compatible producer tables; this notebook does **not** independently reproduce the entire experiment. Keep source-record count separate from sample count, event count and participant count. A conditional vendor agreement is only meaningful beside its evaluable denominator and does not demonstrate detector accuracy or an optimal threshold.

Export `07_evidence_summary.csv` alongside the 40-row record-alignment table. These files document the actual integrated observations and their stated limits without creating new classification labels.


In [ ]:
# ================== 04. Summarise Source-Aligned Evidence =====================

# ------------------- 1. Prepare an Auditable Overview -------------------------
summary = pd.DataFrame([{
    "dataset": "GazeBase v2.0",
    "scope": "Round 1, Session 1; four source tasks",
    "selected_records": len(alignment),
    "independent_participants": independent_people,
    "original_sample_rows": int(quality["n_samples"].sum()),
    "mean_record_valid_fraction": float(quality["valid_fraction"].mean()),
    "n_detector_event_intervals": int(event_counts["n_events"].sum()),
    "n_record_feature_rows": len(features),
    "n_participant_task_rows": len(participants),
    "n_task_feature_statistics": len(stats),
    "records_with_vendor_agreement": int(vendor["sample_agreement"].notna().sum()),
    "n_predefined_ivt_thresholds": len(recomputed_support),
    "source_record_alignment": "validated",
    "vendor_reference_interpretation": "conditional_not_expert_ground_truth",
    "bootstrap_scope": "exploratory_participant_resampling",
    "independent_reproduction": False,
}])
summary.to_csv(output_dir / "07_evidence_summary.csv", index=False)
display(summary.T.rename(columns={0: "Recorded evidence"}))


**05. Visualise the Saved Scientific Evidence**

The three-panel overview presents complementary, *already recorded* results without reprocessing gaze: **(a)** participant-mean valid gaze fraction with Notebook 05 exploratory bootstrap percentiles, **(b)** record-mean I-VT class assignment fractions for the Notebook 06 threshold grid, and **(c)** Notebook 03 algorithm-derived event interval counts.

The quality panel uses a **labelled zoomed y-axis** to keep small interval differences legible. The detector panel is descriptive, and the event counts represent automatically identified intervals rather than verified physiological events. These figures must not be interpreted as a test of task superiority, optimal threshold selection or cognitive performance.


In [ ]:
# ======================= 05. Visualise Saved Evidence ========================

# ---------------- 1. Task Quality and Participant-Bootstrap Limits ------------
validity = stats.loc[stats["feature"].eq("valid_fraction")].sort_values("task")
if validity.empty:
    raise ValueError("Notebook 05 has no valid gaze statistics")
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x = np.arange(len(validity))
axes[0].scatter(x, validity.participant_mean, marker="o", label="Mean", zorder=3)
for i, row in enumerate(validity.itertuples()):
    if np.isfinite(row.bootstrap_ci_lower) and np.isfinite(row.bootstrap_ci_upper):
        axes[0].vlines(i, row.bootstrap_ci_lower, row.bootstrap_ci_upper, lw=2)
        axes[0].hlines(
            [row.bootstrap_ci_lower, row.bootstrap_ci_upper],
            i - 0.08, i + 0.08, lw=1.5,
        )
axes[0].set_xticks(x, validity["task"].tolist())
axes[0].set(
    title="Task Gaze Validity (Zoomed)",
    xlabel="Task", ylabel="Participant mean; bootstrap percentiles",
    ylim=(0.82, 1.02),
)
axes[0].grid(axis="y", alpha=0.2)

# ------------------ 2. Prespecified I-VT Sensitivity --------------------------
axis_thresholds = recomputed_support["threshold_dva_s"]
for field, label in (
    ("mean_predicted_fixation_fraction", "Fixation"),
    ("mean_predicted_saccade_fraction", "Saccade"),
    ("mean_unknown_fraction", "Unknown"),
):
    axes[1].plot(
        axis_thresholds, recomputed_support[field],
        marker="o", label=label,
    )
axes[1].set(
    title="I-VT Threshold Sensitivity",
    xlabel="I-VT threshold (dva/s)", ylabel="Mean fraction across records",
    ylim=(0, 1),
)
axes[1].set_xticks(axis_thresholds)
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.2)

# ------------------ 3. Source-Recorded Detector Intervals ---------------------
event_labels = (
    event_counts["method"].astype(str) + " " +
    event_counts["event_type"].astype(str)
)
axes[2].bar(event_labels, event_counts["n_events"])
axes[2].set(
    title="Algorithm-Derived Event Intervals",
    xlabel="Detector and class", ylabel="Event intervals",
)
axes[2].tick_params(axis="x", rotation=25)
axes[2].grid(axis="y", alpha=0.2)

fig.suptitle("GazeBase S1: Descriptive Evidence Integration")
fig.tight_layout()
save_figure(fig, output_dir / "07_study_overview.png")
plt.show()


**06. Interpret, Save and Transfer the Integrated Evidence**

Notebook 07 owns three current outputs: `07_source_alignment.csv` (record-level provenance checks), `07_evidence_summary.csv` (one row of bounded study coverage) and `07_study_overview.png` (saved descriptive visual synthesis). These are generated **without** rereading the multi-million-row gaze files or recomputing prior scientific stages.

The scientific conclusion is deliberately limited: the selected ten-person GazeBase subset supports a reproducible sequence of source audit, offline preprocessing, provisional event detection, derived features, exploratory participant summaries and conditional threshold sensitivity. Extremely high calculated gaze speeds, vendor taxonomy differences and the small independent cohort prevent stronger interpretation.

Keep named output files and the Notebook 07 ZIP private by default. Completing Notebook 07 will finish the planned seven-notebook execution sequence, but **does not by itself establish scientific validity or authorise repository-wide closure**.


In [ ]:
# ==================== 06. Interpretation and Output Download =================

print("Source records reconciled:", len(alignment))
print("Independent participants:", independent_people)
print("Evidence summary:", output_dir / "07_evidence_summary.csv")
print("Record-alignment evidence:", output_dir / "07_source_alignment.csv")
print("Descriptive overview:", output_dir / "07_study_overview.png")
print("No source gaze processing, fitting or independent replication was performed.")
download_outputs_from_colab("eye_tracking_foundations/output", "07")
